# Battery State of Charge (SOC) Estimation using LSTM

**Author:** [Your Name]
**Date:** [Date]

## Motivation

Accurate State of Charge (SOC) estimation is a core challenge in battery management systems (BMS), particularly relevant to Battery Energy Storage Systems (BESS) used in smart grid and microgrid applications. Traditional methods such as Coulomb counting suffer from drift over time, while model-based approaches (e.g., Kalman filters) require accurate electrochemical models that are difficult to obtain for aging batteries.

This project explores a data-driven approach: using an LSTM (Long Short-Term Memory) neural network to estimate SOC directly from time-series measurements of voltage, current, and temperature. This project was developed as part of my preparation for graduate research in power electronics and battery energy storage systems.

## Dataset

This project uses the **NASA Battery Dataset** (Saha & Goebel, 2007, NASA Prognostics Data Repository), specifically battery **B0005**, which underwent 168 repeated charge/discharge cycles at room temperature (24°C) until end-of-life.


## 1. Setup and Data Loading

Load the raw `.mat` file and inspect its structure.

In [ ]:
import scipy.io
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
import matplotlib.pyplot as plt

MAT_FILE_PATH = "B0005.mat"
BATTERY_NAME = "B0005"


In [ ]:
def load_battery_mat(path, name):
    raw = scipy.io.loadmat(path, simplify_cells=True)
    return raw[name]["cycle"]

cycles = load_battery_mat(MAT_FILE_PATH, BATTERY_NAME)
discharge_idxs = [i for i, c in enumerate(cycles) if c["type"] == "discharge"]

print(f"Total cycles: {len(cycles)}")
print(f"Discharge cycles: {len(discharge_idxs)}")
print(f"Fields available: {list(cycles[discharge_idxs[0]]['data'].keys())}")


## 2. Computing SOC Ground-Truth via Coulomb Counting

The NASA dataset does not provide SOC labels directly. We derive them using **Coulomb counting**:

$$SOC(t) = \frac{C_{total} - \int_0^t I(\tau)\, d\tau}{C_{total}} \times 100\%$$

where $C_{total}$ is the total discharge capacity measured at the end of the cycle, and the integral of current over time gives the charge removed up to time $t$.

**Assumption:** this method assumes the battery reaches 0% SOC at the end of each full discharge cycle, which is standard practice in this type of analysis but represents a simplification worth noting as a limitation.

In [ ]:
def compute_soc_for_cycle(cycle_data, cycle_number):
    time = np.array(cycle_data["Time"]).flatten()
    current = np.array(cycle_data["Current_measured"]).flatten()
    voltage = np.array(cycle_data["Voltage_measured"]).flatten()
    temperature = np.array(cycle_data["Temperature_measured"]).flatten()
    total_capacity = cycle_data["Capacity"]

    time_hours = time / 3600.0
    charge_removed_ah = np.concatenate((
        [0.0],
        np.cumsum((current[1:] + current[:-1]) / 2 * np.diff(time_hours))
    ))
    charge_removed_ah = np.abs(charge_removed_ah)

    remaining_capacity = total_capacity - charge_removed_ah
    soc = np.clip((remaining_capacity / total_capacity) * 100.0, 0, 100)

    return pd.DataFrame({
        "cycle": cycle_number, "time_s": time, "voltage": voltage,
        "current": current, "temperature": temperature, "soc": soc,
    })

all_dfs = []
for cycle_number, idx in enumerate(discharge_idxs):
    all_dfs.append(compute_soc_for_cycle(cycles[idx]["data"], cycle_number))

df = pd.concat(all_dfs, ignore_index=True)
print(f"Total rows: {len(df)}")
print(f"SOC range: {df['soc'].min():.2f}% to {df['soc'].max():.2f}%")
df.head()


In [ ]:
# Visualize one discharge cycle as a sanity check
example = df[df["cycle"] == 0]
fig, axes = plt.subplots(4, 1, figsize=(8, 9), sharex=True)
axes[0].plot(example["time_s"], example["voltage"]); axes[0].set_ylabel("Voltage (V)")
axes[1].plot(example["time_s"], example["current"], color="tab:orange"); axes[1].set_ylabel("Current (A)")
axes[2].plot(example["time_s"], example["temperature"], color="tab:red"); axes[2].set_ylabel("Temp (°C)")
axes[3].plot(example["time_s"], example["soc"], color="tab:green"); axes[3].set_ylabel("SOC (%)")
axes[3].set_xlabel("Time (s)")
axes[0].set_title("Example discharge cycle (cycle 0)")
plt.tight_layout(); plt.show()


## 3. Preprocessing: Normalization and Sliding Windows

- Features (voltage, current, temperature) are scaled to [0, 1].
- Sliding windows of 20 timesteps are built **within each cycle only**, to avoid mixing data across separate discharge sessions.
- Data is split 70/30 into train/test **by cycle** (not by row), preventing information leakage between the two sets.

In [ ]:
WINDOW_SIZE = 20
TRAIN_TEST_SPLIT = 0.7

feature_cols = ["voltage", "current", "temperature"]
mins, maxs = df[feature_cols].min(), df[feature_cols].max()

df_norm = df.copy()
for col in feature_cols:
    df_norm[col] = (df[col] - mins[col]) / (maxs[col] - mins[col])
df_norm["soc"] = df["soc"] / 100.0

def build_windows_for_cycle(cycle_df, window_size):
    features = cycle_df[feature_cols].values
    targets = cycle_df["soc"].values
    X, y = [], []
    n_rows = len(cycle_df)
    if n_rows <= window_size:
        return np.empty((0, window_size, 3)), np.empty((0,))
    for start in range(0, n_rows - window_size):
        end = start + window_size
        X.append(features[start:end])
        y.append(targets[end - 1])
    return np.array(X), np.array(y)

cycle_ids = sorted(df_norm["cycle"].unique())
n_train_cycles = int(len(cycle_ids) * TRAIN_TEST_SPLIT)
train_cycles, test_cycles = cycle_ids[:n_train_cycles], cycle_ids[n_train_cycles:]

def build_all_windows(cycles_subset):
    all_X, all_y = [], []
    for cid in cycles_subset:
        X, y = build_windows_for_cycle(df_norm[df_norm["cycle"] == cid], WINDOW_SIZE)
        if len(X) > 0:
            all_X.append(X); all_y.append(y)
    return np.concatenate(all_X), np.concatenate(all_y)

X_train, y_train = build_all_windows(train_cycles)
X_test, y_test = build_all_windows(test_cycles)

print(f"Train windows: {X_train.shape}, Test windows: {X_test.shape}")


## 4. LSTM Model

A single-layer LSTM (64 hidden units) reads each 20-timestep window and predicts SOC at the final timestep. This simple architecture was chosen after comparing against a deeper 2-layer variant, which overfit and generalized worse on the held-out test set (see Results & Discussion).

In [ ]:
class SOCLSTM(nn.Module):
    def __init__(self, input_size=3, hidden_size=64):
        super().__init__()
        self.lstm = nn.LSTM(input_size=input_size, hidden_size=hidden_size, batch_first=True)
        self.fc = nn.Linear(hidden_size, 1)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        return self.fc(lstm_out[:, -1, :]).squeeze(-1)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=64, shuffle=True)

model = SOCLSTM()
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
model


In [ ]:
NUM_EPOCHS = 20
losses = []
for epoch in range(NUM_EPOCHS):
    model.train()
    epoch_loss = 0.0
    for X_batch, y_batch in train_loader:
        optimizer.zero_grad()
        pred = model(X_batch)
        loss = criterion(pred, y_batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * X_batch.size(0)
    avg_loss = epoch_loss / len(train_loader.dataset)
    losses.append(avg_loss)
    print(f"Epoch {epoch+1}/{NUM_EPOCHS} - Train Loss (MSE): {avg_loss:.6f}")


In [ ]:
plt.figure(figsize=(6,4))
plt.plot(losses)
plt.xlabel("Epoch"); plt.ylabel("Training Loss (MSE)")
plt.title("Training Loss Curve")
plt.tight_layout(); plt.show()


## 5. Evaluation on Test Set

In [ ]:
model.eval()
with torch.no_grad():
    preds = model(X_test_t).numpy()

preds_pct = preds * 100
y_true_pct = y_test_t.numpy() * 100

rmse = np.sqrt(np.mean((preds_pct - y_true_pct) ** 2))
mae = np.mean(np.abs(preds_pct - y_true_pct))
print(f"Test RMSE: {rmse:.3f}%")
print(f"Test MAE:  {mae:.3f}%")


In [ ]:
plt.figure(figsize=(10,5))
plt.plot(y_true_pct[:1000], label="Actual SOC")
plt.plot(preds_pct[:1000], label="Predicted SOC", alpha=0.7)
plt.xlabel("Sample index (test set)"); plt.ylabel("SOC (%)")
plt.title("Predicted vs Actual SOC (first 1000 test samples)")
plt.legend(); plt.tight_layout(); plt.show()


## 6. Results & Discussion

**Quantitative results:** the LSTM model achieved a test RMSE of ~6.2% and MAE of ~4.7% on held-out discharge cycles. RMSE being noticeably higher than MAE indicates that most predictions are fairly accurate, with a subset of larger errors pulling the RMSE up.

**Qualitative results:** the predicted SOC trajectory closely follows the actual sawtooth discharge pattern across cycles, correctly capturing the sharp reset at the start of each new cycle. The main source of error is a systematic underestimation during the mid-discharge "plateau" region, where voltage changes very little despite continued SOC decline — a known challenge in voltage-based SOC estimation, since the voltage-SOC relationship is highly non-linear and flat in this region.

**Model selection:** a deeper 2-layer LSTM variant was also tested, using more training epochs and dropout regularization. While training loss decreased further, test error *increased* (RMSE 7.3%, MAE 6.0%), indicating overfitting. The simpler single-layer model was retained as the final model, since it generalized better to unseen discharge cycles.

**Limitations:**
- SOC ground-truth relies on Coulomb counting, which assumes full discharge to 0% SOC at cycle end and can accumulate small integration errors over time.
- Only a single battery (B0005) was used; generalization to other batteries/chemistries was not evaluated.
- The model does not account for capacity fade over the battery's aging process explicitly as an input feature.

**Relevance to future research:** this SOC estimation approach is directly relevant to Battery Energy Storage Systems (BESS) used in smart grid and microgrid applications, where accurate, real-time SOC estimates are essential for charge/discharge scheduling, state-of-health monitoring, and grid stability control. A natural extension of this work would be integrating this SOC estimator as a component within a broader BESS power converter control simulation, connecting battery-level state estimation to grid-level energy management — a direction I hope to pursue in graduate research.